In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

PROJECT_ROOT = Path("..").resolve()
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "cleaned_transactions.csv"
MCC_FILE = PROJECT_ROOT / "data" / "raw" / "mcc_codes.json"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError("Run src/01_process_data.py before this EDA notebook.")

transactions = pd.read_csv(DATA_FILE)
transactions.shape

## Prepare EDA Columns


In [ ]:
transactions["date"] = pd.to_datetime(transactions["date"], errors="coerce")
transactions["amount_value"] = pd.to_numeric(
    transactions["amount"].astype(str).str.replace("$", "", regex=False),
    errors="coerce",
)
transactions["hour"] = transactions["date"].dt.hour
transactions["day_name"] = transactions["date"].dt.day_name()
transactions["month"] = transactions["date"].dt.to_period("M").astype(str)

transactions[["date", "amount", "amount_value", "hour", "day_name", "is_fraud"]].head()

## Dataset Overview

In [ ]:
overview = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns before EDA features",
            "date start",
            "date end",
            "unique clients",
            "unique cards",
            "unique merchants",
            "fraud transactions",
            "fraud rate (%)",
        ],
        "value": [
            f"{len(transactions):,}",
            transactions.drop(columns=["amount_value", "hour", "day_name", "month"]).shape[1],
            transactions["date"].min(),
            transactions["date"].max(),
            f"{transactions['client_id'].nunique():,}",
            f"{transactions['card_id'].nunique():,}",
            f"{transactions['merchant_id'].nunique():,}",
            f"{int(transactions['is_fraud'].sum()):,}",
            round(transactions["is_fraud"].mean() * 100, 4),
        ],
    }
)
overview

In [ ]:
attribute_types = pd.DataFrame(
    {
        "attribute": [
            "id", "date", "client_id", "card_id", "amount", "use_chip",
            "merchant_id", "merchant_city", "merchant_state", "mcc", "errors", "is_fraud"
        ],
        "EDA type": [
            "identifier", "datetime", "identifier", "identifier", "numerical after parsing", "categorical",
            "identifier", "categorical", "categorical", "categorical code", "error flag/text", "binary target"
        ],
        "use in analysis": [
            "merge key only", "time patterns and features", "customer grouping", "card grouping", "amount distribution", "payment method pattern",
            "merchant grouping", "location pattern", "location pattern", "merchant category pattern", "transaction issue pattern", "fraud label"
        ],
    }
)
attribute_types

## Data Quality After Processing

In [ ]:
missing_summary = (
    transactions.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_percent=lambda table: (table["missing_count"] / len(transactions) * 100).round(2))
    .sort_values("missing_count", ascending=False)
)
missing_summary

In [ ]:
quality_checks = pd.DataFrame(
    {
        "check": [
            "duplicate rows",
            "duplicate transaction ids",
            "invalid parsed dates",
            "invalid parsed amounts",
            "negative or reversal amounts",
        ],
        "value": [
            int(transactions.duplicated().sum()),
            int(transactions["id"].duplicated().sum()),
            int(transactions["date"].isna().sum()),
            int(transactions["amount_value"].isna().sum()),
            int((transactions["amount_value"] < 0).sum()),
        ],
    }
)
quality_checks

## Descriptive Statistics

In [ ]:
transactions[["amount_value", "hour", "is_fraud"]].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).round(2)

In [ ]:
category_summary = {
    "use_chip": transactions["use_chip"].value_counts(dropna=False).head(10),
    "merchant_state": transactions["merchant_state"].value_counts(dropna=False).head(10),
    "mcc": transactions["mcc"].value_counts(dropna=False).head(10),
}
category_summary

## Fraud Distribution


In [ ]:
fraud_counts = transactions["is_fraud"].value_counts().sort_index()
fraud_plot = pd.DataFrame(
    {"label": ["Non-fraud", "Fraud"], "transactions": [fraud_counts.get(0, 0), fraud_counts.get(1, 0)]}
)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=fraud_plot, x="label", y="transactions", hue="label", palette=["#3b82f6", "#dc2626"], legend=False, ax=ax)
ax.set_title("Transaction Count by Fraud Label")
ax.set_xlabel("Fraud label")
ax.set_ylabel("Number of transactions")
ax.ticklabel_format(style="plain", axis="y")
for patch in ax.patches:
    ax.annotate(f"{int(patch.get_height()):,}", (patch.get_x() + patch.get_width() / 2, patch.get_height()), ha="center", va="bottom")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda_fraud_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

fraud_plot.assign(percent=lambda table: (table["transactions"] / len(transactions) * 100).round(4))

## Transaction Amount Patterns



In [ ]:
amount_limit = transactions["amount_value"].quantile(0.99)
amount_for_hist = transactions.loc[
    transactions["amount_value"].between(0, amount_limit, inclusive="both"),
    "amount_value",
]

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(amount_for_hist, bins=60, color="#0f766e", ax=ax)
ax.set_title("Transaction Amount Distribution up to the 99th Percentile")
ax.set_xlabel("Transaction amount")
ax.set_ylabel("Number of transactions")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda_amount_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

pd.DataFrame({"99th percentile amount": [round(amount_limit, 2)], "rows shown": [len(amount_for_hist)]})

In [ ]:
amount_by_fraud = transactions.loc[
    transactions["amount_value"].between(0, amount_limit, inclusive="both"),
    ["amount_value", "is_fraud"],
].copy()
amount_by_fraud["fraud_label"] = amount_by_fraud["is_fraud"].map({0: "Non-fraud", 1: "Fraud"})

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=amount_by_fraud, x="fraud_label", y="amount_value", hue="fraud_label", palette=["#3b82f6", "#dc2626"], legend=False, ax=ax)
ax.set_title("Transaction Amount by Fraud Label up to the 99th Percentile")
ax.set_xlabel("Fraud label")
ax.set_ylabel("Transaction amount")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda_amount_by_fraud.png", dpi=160, bbox_inches="tight")
plt.show()

transactions.groupby("is_fraud")["amount_value"].agg(["count", "mean", "median", "max"]).round(2)

## Time Patterns

In [ ]:
hourly_summary = transactions.groupby("hour", dropna=False).agg(
    transactions=("id", "size"),
    fraud_count=("is_fraud", "sum"),
    fraud_rate=("is_fraud", "mean"),
).reset_index()
hourly_summary["fraud_rate_percent"] = hourly_summary["fraud_rate"] * 100

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.lineplot(data=hourly_summary, x="hour", y="fraud_rate_percent", marker="o", color="#b45309", ax=ax)
ax.set_title("Fraud Rate by Transaction Hour")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Fraud rate (%)")
ax.set_xticks(range(0, 24, 2))
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda_fraud_rate_by_hour.png", dpi=160, bbox_inches="tight")
plt.show()

hourly_summary.head()

## Payment Method and Merchant Category Patterns

In [ ]:
chip_summary = transactions.groupby("use_chip", dropna=False).agg(
    transactions=("id", "size"),
    fraud_count=("is_fraud", "sum"),
    fraud_rate=("is_fraud", "mean"),
).sort_values("transactions", ascending=False).reset_index()
chip_summary["fraud_rate_percent"] = chip_summary["fraud_rate"] * 100

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=chip_summary, x="use_chip", y="fraud_rate_percent", hue="use_chip", palette="Set2", legend=False, ax=ax)
ax.set_title("Fraud Rate by Transaction Method")
ax.set_xlabel("Transaction method")
ax.set_ylabel("Fraud rate (%)")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda_fraud_rate_by_transaction_method.png", dpi=160, bbox_inches="tight")
plt.show()

chip_summary.round(4)

In [ ]:
mcc_summary = transactions.groupby("mcc", dropna=False).agg(
    transactions=("id", "size"),
    fraud_count=("is_fraud", "sum"),
    fraud_rate=("is_fraud", "mean"),
).reset_index()
mcc_summary = mcc_summary[mcc_summary["transactions"] >= 500].copy()
mcc_summary["fraud_rate_percent"] = mcc_summary["fraud_rate"] * 100

if MCC_FILE.exists():
    with MCC_FILE.open() as handle:
        mcc_names = json.load(handle)
    mcc_summary["mcc_name"] = mcc_summary["mcc"].astype(str).map(mcc_names)
else:
    mcc_summary["mcc_name"] = pd.NA

top_mcc = mcc_summary.sort_values("fraud_rate_percent", ascending=False).head(10).copy()
top_mcc["mcc_label"] = top_mcc["mcc"].astype(str) + " - " + top_mcc["mcc_name"].fillna("Unknown")

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(data=top_mcc, y="mcc_label", x="fraud_rate_percent", hue="mcc_label", palette="rocket", legend=False, ax=ax)
ax.set_title("Top Merchant Categories by Fraud Rate (Minimum 500 Transactions)")
ax.set_xlabel("Fraud rate (%)")
ax.set_ylabel("Merchant category code")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda_top_mcc_fraud_rate.png", dpi=160, bbox_inches="tight")
plt.show()

top_mcc[["mcc", "mcc_name", "transactions", "fraud_count", "fraud_rate_percent"]].round(4)

## Numeric Correlation Check


In [ ]:
correlation_columns = ["amount_value", "hour", "mcc", "is_fraud"]
corr = transactions[correlation_columns].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation Heatmap for Selected Numeric EDA Features")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "eda_selected_feature_correlation.png", dpi=160, bbox_inches="tight")
plt.show()

corr.round(3)